# Evaluación del Modelo - BLEU Score y Análisis

**Fase 4 (Evaluation) + Fase 6 (Monitoring)**

Evaluación completa del modelo entrenado.
Cálculo de BLEU Score, análisis de errores y ejemplos de traducción.

In [ ]:
import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '/home/honorio/IA/rnn/src')

from metrics import BLEUScore, ErrorAnalyzer, exact_match_accuracy
from preprocessing import ParallelDataProcessor
from inference import TranslationInference

base_path = Path('/home/honorio/IA/rnn')
splits_path = base_path / 'data' / 'splits'
processed_path = base_path / 'data' / 'processed'
models_path = base_path / 'models'

print("✓ Setup completado")

## 1. Cargar Datos y Modelo

In [ ]:
# Cargar datos de prueba
X_test = np.load(splits_path / 'X_test.npy')
y_test = np.load(splits_path / 'y_test.npy')

# Cargar vocabularios
with open(processed_path / 'vocab_spanish.json', 'r') as f:
    vocab_spanish = json.load(f)
with open(processed_path / 'vocab_english.json', 'r') as f:
    vocab_english = json.load(f)

# Cargar modelo
model = tf.keras.models.load_model(models_path / 'traductor_v1')
print(f"✓ Modelo cargado")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")
print(f"Vocabularios: ES={len(vocab_spanish)}, EN={len(vocab_english)}")

## 2. Calcular BLEU Score

In [ ]:
# Realizar predicciones
idx2spanish = {v: k for k, v in vocab_spanish.items()}
idx2english = {v: k for k, v in vocab_english.items()}

bleu_calculator = BLEUScore()

# Generar predicciones (primeras N muestras para evaluación)
predictions = model.predict(X_test, verbose=0)
predicted_indices = np.argmax(predictions, axis=-1)

# Decodificar predicciones y referencias
predicted_sentences = []
reference_sentences = []

for i in range(min(100, len(y_test))):  # Evaluar primeras 100
    pred_tokens = [idx2english.get(int(idx), '<UNK>') for idx in predicted_indices[i]]
    pred_tokens = [t for t in pred_tokens if t not in ['<PAD>', '<START>', '<END>', '<UNK>']]
    predicted_sentences.append(pred_tokens)
    
    ref_tokens = [idx2english.get(int(idx), '<UNK>') for idx in y_test[i]]
    ref_tokens = [t for t in ref_tokens if t not in ['<PAD>', '<START>', '<END>', '<UNK>']]
    reference_sentences.append(ref_tokens)

# Calcular BLEU
bleu = bleu_calculator.compute_corpus_bleu(predicted_sentences, reference_sentences)
exact_match = exact_match_accuracy(predicted_sentences, reference_sentences)

print(f"📊 RESULTADOS EVALUACIÓN")
print(f"  BLEU Score: {bleu:.4f}")
print(f"  Exact Match: {exact_match:.4f}")

# Guardar resultados
eval_results = {
    'bleu_score': float(bleu),
    'exact_match': float(exact_match),
    'samples_evaluated': 100,
    'timestamp': str(pd.Timestamp.now())
}

with open(models_path / 'evaluation_metrics.json', 'w') as f:
    json.dump(eval_results, f, indent=2)

print(f"✓ Resultados guardados en models/evaluation_metrics.json")

## 3. Ejemplos de Traducción

In [ ]:
print("📝 EJEMPLOS DE TRADUCCIÓN")
print("=" * 80)

for i in range(min(10, len(predicted_sentences))):
    pred = ' '.join(predicted_sentences[i])
    ref = ' '.join(reference_sentences[i])
    
    print(f"\nEjemplo {i+1}:")
    print(f"  Predicción: {pred}")
    print(f"  Referencia: {ref}")

print("\n✓ Evaluación completada")